# 🏋️ Workout Log — __DATE__

Fill in your session below. Run the last cell when done to export JSON + MD.

In [ ]:
# ── Imports (do not edit)
import json
import os
from IPython.display import Image, display, Markdown
from datetime import datetime


---
## 📋 Session Info

In [ ]:
# ── Session metadata
session = {
    "date": "__DATE__",
    "day": datetime.strptime("__DATE__", "%Y-%m-%d").strftime("%A"),  # auto
    "bodyweight_kg": 75,          # ← your bodyweight today
    "duration_min": 60,           # ← total gym time in minutes
    "overall_feel": "good",       # ← great / good / average / bad
    "notes": ""                   # ← anything about the session overall
}

print(f"📅 {session['day']}, {session['date']}")
print(f"⚖️  Bodyweight: {session['bodyweight_kg']} kg")
print(f"⏱️  Duration: {session['duration_min']} min")
print(f"💬 Feel: {session['overall_feel']}")


---
## 💪 Exercises

> Copy the block below for each exercise. Delete the ones you don't use.

In [ ]:
# ── EXERCISE TEMPLATE (reference — do not run this cell for real data)
# {
#     "exercise": "Exercise Name",
#     "muscle_group": "chest | back | shoulders | biceps | triceps | legs | core | cardio",
#     "category": "strength",      # strength | cardio | flexibility
#     "sets": [
#         {"set": 1, "reps": 10, "weight_kg": 60},
#         {"set": 2, "reps": 8,  "weight_kg": 65},
#         {"set": 3, "reps": 6,  "weight_kg": 70},
#     ],
#     "notes": "How it felt, form notes, PR, etc."
# }
#
# ── CARDIO TEMPLATE
# {
#     "exercise": "Treadmill Run",
#     "muscle_group": "cardio",
#     "category": "cardio",
#     "cardio": {
#         "duration_min": 30,
#         "distance_km": 4.5,
#         "calories": 320
#     },
#     "notes": ""
# }
print("📖 Template reference loaded. Start filling exercises below.")


In [ ]:
# ── YOUR EXERCISES — fill these in
exercises = [

    {
        "exercise": "Bench Press",
        "muscle_group": "chest",
        "category": "strength",
        "sets": [
            {"set": 1, "reps": 10, "weight_kg": 60},
            {"set": 2, "reps": 8,  "weight_kg": 65},
            {"set": 3, "reps": 6,  "weight_kg": 70},
        ],
        "notes": "Felt strong. Shoulder held up fine."
    },

    {
        "exercise": "Incline Dumbbell Press",
        "muscle_group": "chest",
        "category": "strength",
        "sets": [
            {"set": 1, "reps": 12, "weight_kg": 20},
            {"set": 2, "reps": 10, "weight_kg": 22},
            {"set": 3, "reps": 8,  "weight_kg": 24},
        ],
        "notes": ""
    },

    {
        "exercise": "Tricep Pushdown",
        "muscle_group": "triceps",
        "category": "strength",
        "sets": [
            {"set": 1, "reps": 15, "weight_kg": 25},
            {"set": 2, "reps": 12, "weight_kg": 27},
            {"set": 3, "reps": 10, "weight_kg": 30},
        ],
        "notes": "Good pump"
    },

    # ── add more exercises here

]

print(f"✅ {len(exercises)} exercises loaded")
for e in exercises:
    if e["category"] == "cardio":
        c = e.get("cardio", {})
        print(f"  🏃 {e['exercise']} — {c.get('distance_km','?')}km, {c.get('calories','?')} cal")
    else:
        total_vol = sum(s["reps"] * s["weight_kg"] for s in e.get("sets", []))
        print(f"  💪 {e['exercise']} ({e['muscle_group']}) — {len(e['sets'])} sets | vol: {total_vol} kg")


---
## 🖼️ Muscle Groups Worked

In [ ]:
# ── Show muscle images for groups worked today
DATE_STR = "__DATE__"
YEAR, MONTH, DAY = DATE_STR.split("-")
ASSETS_PATH = os.path.join("..", "..", "..", "..", "assets", YEAR, MONTH, DAY)

muscle_groups_today = list(set(
    e["muscle_group"] for e in exercises
    if e["muscle_group"] != "cardio"
))

print(f"Muscle groups today: {', '.join(muscle_groups_today)}")
print()

for mg in muscle_groups_today:
    for ext in [".png", ".jpg", ".jpeg", ".webp"]:
        img_path = os.path.join(ASSETS_PATH, mg + ext)
        if os.path.exists(img_path):
            display(Markdown(f"### {mg.title()}"))
            display(Image(filename=img_path, width=300))
            break
    else:
        display(Markdown(f"### {mg.title()} _(no image yet — add {mg}.png to assets/muscles/)_"))


---
## 💾 Export — Run This Last

In [ ]:
# ── Export: generates workout.json and workout.md in the same folder
# Run this cell after you've filled in everything above.

THIS_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()

# ── Resolve image paths for each muscle group
DATE_STR  = "__DATE__"
YEAR, MONTH, DAY = DATE_STR.split("-")
ASSETS_PATH = os.path.join("..", "..", "..", "..", "assets", YEAR, MONTH, DAY)
# Relative path used inside MD (from logs/YYYY/MM/DD/ → assets/YYYY/MM/DD/)
ASSETS_REL  = os.path.join("..", "..", "..", "..", "assets", YEAR, MONTH, DAY)

def find_image(muscle_group):
    """Returns (abs_path, rel_path) if image exists, else (None, None)."""
    for ext in [".png", ".jpg", ".jpeg", ".webp"]:
        abs_p = os.path.join(ASSETS_PATH, muscle_group + ext)
        rel_p = os.path.join(ASSETS_REL,  muscle_group + ext).replace("\\", "/")
        if os.path.exists(abs_p):
            return abs_p, rel_p
    return None, None

# ── Attach image_path to each exercise
exercises_with_images = []
for e in exercises:
    entry = dict(e)
    _, rel_p = find_image(e["muscle_group"])
    entry["image_path"] = rel_p  # None if no image found
    exercises_with_images.append(entry)

# ── Build full data object
data = {
    **session,
    "exercises": exercises_with_images
}

# ── 1. Write JSON
json_path = os.path.join(THIS_DIR, "workout.json")
with open(json_path, "w") as f:
    json.dump(data, f, indent=2)
print(f"✅ JSON saved → {json_path}")

# ── 2. Write Markdown
md_lines = []
md_lines.append(f"# 🏋️ Workout — {data['date']} ({data['day']})")
md_lines.append("")
md_lines.append(f"**Bodyweight:** {data['bodyweight_kg']} kg  ")
md_lines.append(f"**Duration:** {data['duration_min']} min  ")
md_lines.append(f"**Feel:** {data['overall_feel']}  ")
if data.get("notes"):
    md_lines.append(f"**Notes:** {data['notes']}  ")
md_lines.append("")
md_lines.append("---")
md_lines.append("")

seen_images = set()
for e in exercises_with_images:
    md_lines.append(f"## {e['exercise']} `{e['muscle_group']}`")
    # ── show muscle image once per muscle group
    if e["image_path"] and e["muscle_group"] not in seen_images:
        md_lines.append(f"![{e['muscle_group']}]({e['image_path']})")
        seen_images.add(e["muscle_group"])
    md_lines.append("")
    if e["category"] == "cardio":
        c = e.get("cardio", {})
        md_lines.append(f"- Duration: {c.get('duration_min', '—')} min")
        md_lines.append(f"- Distance: {c.get('distance_km', '—')} km")
        md_lines.append(f"- Calories: {c.get('calories', '—')}")
    else:
        md_lines.append("| Set | Reps | Weight (kg) |")
        md_lines.append("|-----|------|-------------|")
        for s in e.get("sets", []):
            md_lines.append(f"| {s['set']}   | {s['reps']}   | {s['weight_kg']}          |")
        total_vol = sum(s["reps"] * s["weight_kg"] for s in e.get("sets", []))
        md_lines.append("")
        md_lines.append(f"**Total Volume:** {total_vol} kg")
    if e.get("notes"):
        md_lines.append("")
        md_lines.append(f"_{e['notes']}_")
    md_lines.append("")

md_path = os.path.join(THIS_DIR, "workout.md")
with open(md_path, "w") as f:
    f.write("\n".join(md_lines))
print(f"✅ Markdown saved → {md_path}")

print()
print("── Done! Now run:")
print(f'   git add logs/__DATE__ assets/__DATE__')
print(f'   git commit -m "log: __DATE__ — {", ".join(set(e["muscle_group"] for e in exercises))}')
print( "   git push")
